In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, classification_report, confusion_matrix

# 1. Load Data
df = pd.read_csv('../data/credit_risk_dataset.csv')

# 2. Clean Outliers
df = df[df['person_age'] <= 90]
df['person_emp_length'] = df['person_emp_length'].fillna(0)
df = df[df['person_emp_length'] <= df['person_age']]

In [2]:
# 1. Create Logic Flags
df['employment_status'] = np.where(df['person_emp_length'] > 0, 'EMPLOYED', 'UNEMPLOYED')
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'N': 0, 'Y': 1})

# 2. Drop useless/leaky columns
df = df.drop(['loan_grade', 'person_emp_length'], axis=1)

# 3. Translate Categories (The Light Switch Method)
df_final = pd.get_dummies(df, columns=['person_home_ownership', 'loan_intent', 'employment_status'], 
                          drop_first=False, 
                          dtype=int)

In [3]:
# 1. Isolate known vs missing
known_rates = df_final[df_final['loan_int_rate'].notnull()]
missing_rates = df_final[df_final['loan_int_rate'].isnull()]

# 2. Define the "Blindfold" (Hide the answer key)
allowed_features = known_rates.drop(['loan_status', 'loan_int_rate'], axis=1).columns

# 3. Train the Imputer
xgb_imputer = XGBRegressor(random_state=42)
xgb_imputer.fit(known_rates[allowed_features], known_rates['loan_int_rate'])

# 4. Predict and Inject
df_final.loc[df_final['loan_int_rate'].isnull(), 'loan_int_rate'] = xgb_imputer.predict(missing_rates[allowed_features])

In [4]:
# 1. The Vault (Split Data)
X = df_final.drop('loan_status', axis=1)
y = df_final['loan_status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Calculate Punishment Multiplier
imbalance_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

# 3. Train the Model
xgb_model = XGBClassifier(scale_pos_weight=imbalance_ratio, random_state=50)
xgb_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [9]:
# 1. Get raw percentages
y_probabilities = xgb_model.predict_proba(X_test)[:, 1]

# 2. Find optimal threshold for F1
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probabilities)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls)
best_threshold = thresholds[np.argmax(f1_scores)]

# 3. Make Final Decisions
optimized_predictions = (y_probabilities >= best_threshold).astype(int)
print(f"Optimal Threshold: {best_threshold:.3f}")
print(classification_report(y_test, optimized_predictions))

Optimal Threshold: 0.643
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      5084
           1       0.88      0.74      0.80      1431

    accuracy                           0.92      6515
   macro avg       0.90      0.86      0.88      6515
weighted avg       0.92      0.92      0.92      6515



In [23]:
import requests
import pandas as pd

# Helper function to format Indian currency properly (e.g., 80,00,000)
def format_inr(number):
    s, *d = str(int(number)).partition(".")
    r = ",".join([s[x-2:x] for x in range(-3, -len(s), -2)][::-1] + [s[-3:]])
    return "".join([r] + d)

def process_loan_application(customer_data, trained_xgb_model, trained_imputer, imputer_features, trained_columns, best_threshold):
    
    input_df = pd.DataFrame([customer_data])
    model_input = pd.DataFrame(columns=trained_columns)
    model_input.loc[0] = 0 
    
    intent = input_df['loan_intent'][0].upper()
    income = input_df['person_income'][0]
    requested_loan = input_df['loan_amnt'][0]
    has_default = input_df['cb_person_default_on_file'][0]
    
    # Map basic inputs
    model_input['person_age'] = input_df['person_age']
    model_input['person_income'] = income
    model_input['cb_person_default_on_file'] = has_default
    
    home_col = f"person_home_ownership_{input_df['person_home_ownership'][0].upper()}"
    if home_col in model_input.columns: model_input[home_col] = 1
        
    intent_col = f"loan_intent_{intent}"
    if intent_col in model_input.columns: model_input[intent_col] = 1
        
    emp_status = 'EMPLOYED' if input_df['person_emp_length'][0] > 0 else 'UNEMPLOYED'
    emp_col = f"employment_status_{emp_status}"
    if emp_col in model_input.columns: model_input[emp_col] = 1

    # ==========================================
    # STEP 2: BUSINESS RULES ENGINE (THE VAULT)
    # ==========================================
    is_tailored = False
    approved_loan_amnt = requested_loan
    status = ""
    calculated_rate = 0.0

    # BRE RULE 1: Zero Tolerance for Past Defaults
    if has_default == 1:
        print("--- BRE INTERVENTION: Auto-Rejected due to Prior Default History. ---")
        status = "REJECTED"
        approved_loan_amnt = 0
        
    # BRE RULE 2: Zero Income Auto-Reject
    elif income <= 0:
        print("--- BRE INTERVENTION: Auto-Rejected due to 0 income. ---")
        status = "REJECTED"
        approved_loan_amnt = 0
        
    else:
        # BRE RULE 3: The Tailoring Logic
        if intent == "HOME": max_multiplier = 5.0
        elif intent == "EDUCATION": max_multiplier = 3.0
        else: max_multiplier = 1.0 
            
        max_allowed_loan = income * max_multiplier
        
        if requested_loan > max_allowed_loan:
            print(f"--- BRE INTERVENTION: Tailoring to Max Limit: ₹{max_allowed_loan}. ---")
            approved_loan_amnt = max_allowed_loan
            is_tailored = True
            model_input['loan_percent_income'] = max_multiplier
        else:
            model_input['loan_percent_income'] = requested_loan / income

        model_input['loan_amnt'] = approved_loan_amnt

        # ==========================================
        # STEP 3: DYNAMIC PRICING (THE CLAMP)
        # ==========================================
        base_rate = trained_imputer.predict(model_input[imputer_features])[0]
        
        if intent == "HOME": calculated_rate = max(7.10, min(10.50, base_rate)) 
        elif intent == "EDUCATION": calculated_rate = max(4.00, min(16.00, base_rate)) 
        else: calculated_rate = min(24.00, base_rate)
            
        model_input['loan_int_rate'] = calculated_rate

        # ==========================================
        # STEP 4: XGBOOST RISK PREDICTION
        # ==========================================
        prob = trained_xgb_model.predict_proba(model_input)[:, 1][0]
        if prob >= best_threshold:
            status = "REJECTED"
            is_tailored = False # FIX: If XGBoost rejects, cancel the "tailoring" offer completely!
        else:
            status = "APPROVED"

    # ==========================================
    # STEP 5: OLLAMA GEN AI EMAIL
    # ==========================================
    # Formatting fixes to hide 0% bug and format Rupees
    display_rate = f"{calculated_rate:.2f}%" if calculated_rate > 0 else "N/A"
    display_requested = format_inr(requested_loan)
    display_approved = format_inr(approved_loan_amnt)

    prompt = f"""
    You are a Senior Loan Officer at a prestigious Indian bank. 
    Write an email to a customer regarding their recent loan application.

    Customer Profile:
    - Name: {customer_data['name']}
    - Requested Rate: {customer_data.get('loan_int_rate', 'Not specified')}%
    - Assigned Rate: {display_rate}
    - Original Requested Amount: ₹{display_requested}
    - Final Approved Amount: ₹{display_approved}
    - Was Amount Tailored Down?: {is_tailored}
    
    Bank Decision: {status}
    
    Instructions:
    - CRITICAL: DO NOT explicitly state the customer's numerical income.
    - CRITICAL: DO NOT mention the "Assigned Rate" if the Bank Decision is REJECTED.
    - If APPROVED AND WAS NOT TAILORED: Be warm and congratulatory. Confirm they are approved for ₹{display_approved}. Compare their Requested Rate to the Assigned Rate.
    - If APPROVED AND WAS TAILORED DOWN: Offer a "Conditional Approval". Explain that to ensure sustainable repayment, we are offering a tailored maximum loan of ₹{display_approved} at {display_rate}.
    - If REJECTED: Be empathetic but firm. State that based on current risk criteria and underwriting guidelines, we cannot approve the loan. DO NOT mention any alternative loan amounts or interest rates.
    - Sign off as "The Risk Analytics Team". Keep it under 150 words.
    """

    try:
        response = requests.post('http://localhost:11434/api/generate',
            json={"model": "llama3", "prompt": prompt, "stream": False}
        )
        response.raise_for_status() 
        return response.json()['response']
    except requests.exceptions.RequestException as e:
        return f"Error connecting to Ollama: {e}"
    
    
# ==========================================
# HOW TO RUN IT
# ==========================================
print("=== WELCOME TO THE SPRINGSTER RISK ENGINE ===")
print("Please enter the customer details below:\n")

new_customer = {
    "name": input("Customer Name: "),
    "person_age": int(input("Age: ")),
    "person_income": int(input("Annual Income (INR): ")),
    "person_home_ownership": input("Housing Status (RENT/OWN/MORTGAGE): ").upper(),
    "person_emp_length": float(input("Years of Employment: ")),
    "loan_intent": input("Loan Intent (MEDICAL/PERSONAL/EDUCATION/VENTURE/HOME): ").upper(),
    "loan_amnt": int(input("Loan Amount Requested (INR): ")),
    "loan_int_rate": float(input("Current Interest Rate (e.g., 10.5): ")),
    "cb_person_default_on_file": int(input("Prior Default? (1 for Yes, 0 for No): "))
}

# Run the updated Engine
final_email = process_loan_application(
    customer_data = new_customer, 
    trained_xgb_model = xgb_model, 
    trained_imputer = xgb_imputer,             
    imputer_features = allowed_features,       
    trained_columns = X_train.columns,
    best_threshold = best_threshold 
)
print(new_customer)
print("\n" + "="*50 + "\nFINAL EMAIL TO CUSTOMER:\n" + "="*50)
print(final_email)

=== WELCOME TO THE SPRINGSTER RISK ENGINE ===
Please enter the customer details below:

--- BRE INTERVENTION: Tailoring to Max Limit: ₹600000.0. ---
{'name': 'Arjun', 'person_age': 20, 'person_income': 200000, 'person_home_ownership': 'RENT', 'person_emp_length': 0.5, 'loan_intent': 'EDUCATION', 'loan_amnt': 5000000, 'loan_int_rate': 5.0, 'cb_person_default_on_file': 0}

FINAL EMAIL TO CUSTOMER:
Subject: Loan Application Update for Arjun

Dear Arjun,

We appreciate your interest in securing a loan from our esteemed institution. After careful review and analysis of your application, we regret to inform you that we are unable to approve the loan as requested.

Our risk assessment indicates that the loan amount and terms you sought do not align with our current underwriting guidelines. As a result, we cannot proceed with the original loan application.

We understand this decision may be disappointing, but please be assured that we have taken your financial situation into careful considera